# Lesson 29 Lab — Security and Compliance Boundaries

**Puzzle:** Can an authenticated generation request still reach private infrastructure or leak sensitive data?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Authentication identifies a caller; it does not make remote media URLs, local model paths, custom code, prompts, logs, adapters, or generated tool arguments safe. Every input channel needs a trust and retention decision.


## 0. Predict before running

1. Classify each URL fixture.
2. Find which data-policy fields are missing.
3. Write one release blocker for remote code or model license.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab evaluates a URL allowlist/SSRF policy against public, loopback, link-local, private, malformed, and redirect-like cases; it also lints a release data-policy manifest for secrets, prompt logging, and model-license fields.

- Authentication and input safety are independent layers.
- DNS/redirect revalidation is required after the initial string check.
- Observability must not silently become indefinite prompt storage.


## 2. Derive the mechanism

SSRF defenses parse the URL, resolve all addresses, reject non-HTTP schemes and private/link-local/loopback ranges, revalidate redirects, and constrain size/content. Prompt and response data need collection purpose, encryption, retention, deletion, and access policy. Model licenses and `trust_remote_code` are supply-chain controls rather than request filters.

### Mechanism at a glance

```mermaid
flowchart TD
  R["authenticated request"] --> I{"input channel"}
  I --> U["URL parse + DNS/IP + redirect policy"]
  I --> P["prompt/data retention policy"]
  I --> A["adapter/model provenance"]
  U --> E["bounded engine request"]
  P --> E
  A --> E
  E --> L["minimized audit record"]
```

### Walk it step by step

1. **Enumerate input channels.** Include URLs, files, prompts, adapters, schemas, and custom code.
2. **Validate after resolution.** Reject unsafe schemes/addresses and re-check redirects.
3. **Minimize data.** Collect only what has a purpose, retention, deletion, and access rule.
4. **Gate the supply chain.** Pin model/adapter bytes, licenses, and executable trust.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 29
LESSON_TITLE = 'Security and Compliance Boundaries'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260841
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | accept authenticated URLs and log full requests |
| Candidate | allowlisted destinations, resolved-IP controls, bounded media, minimized logs, and provenance gates |
| Held constant | fixture URLs, simulated DNS map, manifest schema, no external fetch, and GPU identity |
| Measurements | allowed/blocked cases, false decisions, policy checks, retention days, and release blockers |
| Evidence | `numerical-model` |

**Experiment:** Run deterministic SSRF-policy fixtures and lint a serving data/supply-chain manifest.


## 5. Inspect the experiment code

The URL test never performs network requests; a frozen DNS map makes the policy auditable and safe. The manifest linter names every missing control.

Do not execute until the code matches the frozen table.


In [2]:
dns={"public.example":["93.184.216.34"],"localhost.example":["127.0.0.1"],
     "metadata.example":["169.254.169.254"],"private.example":["10.2.3.4"],"v6local.example":["::1"]}
fixtures=[("https://public.example/image.png",True),("http://private.example/a",False),
 ("http://metadata.example/latest",False),("http://localhost.example/a",False),
 ("file:///etc/passwd",False),("gopher://public.example/x",False),("http://v6local.example/x",False)]
def allow(url):
    parsed=urlparse(url)
    if parsed.scheme not in {"http","https"} or not parsed.hostname: return False
    addresses=dns.get(parsed.hostname,[])
    return bool(addresses) and all(not (ipaddress.ip_address(a).is_private or ipaddress.ip_address(a).is_loopback
        or ipaddress.ip_address(a).is_link_local or ipaddress.ip_address(a).is_reserved) for a in addresses)
decisions=[{"url":url,"expected":expected,"actual":allow(url)} for url,expected in fixtures]
policy={"api_keys_in_secret_store":True,"full_prompt_metric_labels":False,"prompt_log_retention_days":7,
 "deletion_workflow":True,"trust_remote_code":False,"model_license_reviewed":True,"redirect_revalidation":True}
checks={"secret_store":policy["api_keys_in_secret_store"],"no_prompt_labels":not policy["full_prompt_metric_labels"],
 "bounded_retention":0<=policy["prompt_log_retention_days"]<=30,"deletion":policy["deletion_workflow"],
 "remote_code_disabled":not policy["trust_remote_code"],"license_review":policy["model_license_reviewed"],
 "redirect_revalidation":policy["redirect_revalidation"]}
errors=sum(x["expected"]!=x["actual"] for x in decisions)
metrics={"url_cases":len(decisions),"url_decisions_correct":len(decisions)-errors,"decisions":decisions,
 "private_blocked":not allow("http://private.example/a"),"link_local_blocked":not allow("http://metadata.example/latest"),
 "policy":policy,"policy_checks":checks,"policy_checks_passed":sum(checks.values()),"policy_checks_total":len(checks),
 "release_blockers":errors+sum(not x for x in checks.values())}
analysis=(f"The SSRF policy classified {metrics['url_decisions_correct']}/{len(decisions)} fixtures and "
          f"passed {metrics['policy_checks_passed']}/{len(checks)} data/supply-chain checks, leaving "
          f"{metrics['release_blockers']} blockers. Real DNS/redirect tests remain required.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| URL cases | 7 |
| URL decisions correct | 7 |
| Private blocked | yes |
| Link-local blocked | yes |
| Policy checks passed | 7 |
| Policy checks total | 7 |
| Release blockers | 0 |


## 7. Explain the result

The SSRF policy classified 7/7 fixtures and passed 7/7 data/supply-chain checks, leaving 0 blockers. Real DNS/redirect tests remain required.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. A transparent allocator, scheduler, gateway, or policy model executed. It establishes the stated invariant, not native vLLM performance.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 29, "title": 'Security and Compliance Boundaries', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Authenticated inference still needs strict input, supply-chain, and data-lifecycle controls; the lab verifies policy logic, not legal compliance.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 29,
  "title": "Security and Compliance Boundaries",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260841
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "url_cases": 7,
    "url_decisions_correct": 7,
    "decisions": [
      {
        "url": "https://public.example/image.png",
        "expected": true,
        "actual": true
      },
      {
        "url": "http://private.example/a",
        "expected": false,
        "actual": false
      },
      {
        "url": "http://metadata.example/latest",
        "expected": false,
        "actual": false
      },
      {
        "url": "http://localhost.example/a",
        "expected": false,
        "actual": false
      },
      {
        "url": "file:///etc/passwd",
        "expected": false,
        "actual

## 9. Make the bounded decision

> Authenticated inference still needs strict input, supply-chain, and data-lifecycle controls; the lab verifies policy logic, not legal compliance.

**Acceptance/rollback:** Block release until input channels, secrets, remote code, model/license provenance, data retention, deletion, and incident ownership are approved and tested.

**Failure analysis:** A simulated resolver cannot expose DNS rebinding, proxy behavior, parser inconsistencies, decompression bombs, or real redirect chains. Compliance requirements are jurisdiction- and organization-specific.


## 10. Extend the evidence

Test the gateway fetcher in an isolated network with redirect/rebinding fixtures, malware/media limits, audit access, deletion workflows, and legal review.

The full boundary and references are in [`README.md`](README.md).
